# 02 · Candidate Generation, Features and Labels

Build candidate/feature tables for the train and student-test partitions.

Outputs (data/processed/):
  {split}_cands.pkl   one row per (trip, point, candidate arc) with features + label
  {split}_points.pkl  one row per GPS point with its reference arc and label distance
Also writes results/candidate_radius_sweep.csv (miss rate vs radius, TRAIN only).

In [1]:
import sys, time
from pathlib import Path
ROOT = Path.cwd().resolve().parent          # run this notebook from the notebooks/ folder
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from pathsnap import config as C
from pathsnap.data_io import load_trip, list_trips
from pathsnap.candidates import generate_candidates, build_features
from pathsnap.labels import align_points_to_route

In [2]:
def process_split(split, split_dir):
    rows, pts = [], []
    for d in list_trips(split_dir):
        t0 = time.time()
        trip = load_trip(d)
        lab_arc, lab_pos, lab_dist = align_points_to_route(trip)
        cands = generate_candidates(trip)
        feats = build_features(trip, cands)
        feats["label"] = (feats.arc.to_numpy() == lab_arc[feats.pt.to_numpy()]).astype(int)
        feats.insert(0, "trip", trip.trip_id)
        rows.append(feats)
        has = np.zeros(len(trip.gps_xy), bool)
        has[feats.loc[feats.label == 1, "pt"].to_numpy()] = True
        ncand = np.bincount(feats.pt, minlength=len(trip.gps_xy))
        pts.append(pd.DataFrame({"trip": trip.trip_id, "pt": np.arange(len(trip.gps_xy)),
                                 "ref_arc": lab_arc, "ref_pos": lab_pos, "ref_dist": lab_dist,
                                 "n_cand": ncand, "truth_in_cands": has,
                                 "t": trip.track[:, 2]}))
        print(f"{split} {trip.trip_id}: pts={len(trip.gps_xy)} cand_rows={len(feats)} "
              f"miss={1-has.mean():.3f} {time.time()-t0:.1f}s", flush=True)
    return pd.concat(rows, ignore_index=True), pd.concat(pts, ignore_index=True)

In [3]:
def radius_sweep():
    out = []
    for r in [20, 30, 50, 75, 100]:
        miss, n, ncs = 0, 0, []
        for d in list_trips(C.TRAIN_DIR):
            trip = load_trip(d)
            lab_arc, _, lab_d = align_points_to_route(trip)
            rel = lab_d <= C.LABEL_RELIABLE_M          # only points with a trustworthy label
            c = generate_candidates(trip, radius=r, k=50)
            hit = c[c.arc.to_numpy() == lab_arc[c.pt.to_numpy()]].pt.unique()
            hit = hit[rel[hit]]
            miss += rel.sum() - len(hit); n += rel.sum()
            ncs.append(c.groupby("pt").size().mean())
        out.append({"radius_m": r, "candidate_miss_rate": miss / n, "mean_candidates": float(np.mean(ncs))})
        print(out[-1], flush=True)
    pd.DataFrame(out).to_csv(C.RESULTS / "candidate_radius_sweep.csv", index=False)

In [4]:
C.PROCESSED.mkdir(parents=True, exist_ok=True); C.RESULTS.mkdir(exist_ok=True)
RUN_SWEEP = False  # set True to re-run the candidate-radius sweep (~5 min); saved result: results/candidate_radius_sweep.csv
if RUN_SWEEP:
    radius_sweep()
for split, sd in [("train", C.TRAIN_DIR), ("student_test", C.STUDENT_TEST_DIR)]:
    cands, pts = process_split(split, sd)
    cands.to_pickle(C.PROCESSED / f"{split}_cands.pkl")
    pts.to_pickle(C.PROCESSED / f"{split}_points.pkl")
    print(split, "rows", len(cands), "points", len(pts), "pos rate", cands.label.mean(),
          "miss", 1 - pts.truth_in_cands.mean())

train 00000001: pts=1315 cand_rows=11311 miss=0.001 1.7s


train 00000002: pts=7243 cand_rows=59060 miss=0.120 0.4s


train 00000003: pts=821 cand_rows=9155 miss=0.002 0.2s


train 00000004: pts=1064 cand_rows=6917 miss=0.005 0.2s


train 00000005: pts=1045 cand_rows=3124 miss=0.000 0.0s


train 00000006: pts=1450 cand_rows=11362 miss=0.000 0.1s


... (82 outputs left out)

student_test 00000076: pts=1486 cand_rows=15880 miss=0.002 0.6s


student_test 00000077: pts=3911 cand_rows=36752 miss=0.000 0.3s


student_test 00000091: pts=874 cand_rows=8196 miss=0.000 0.2s


student_test rows 147062 points 19160 pos rate 0.13000639186193577 miss 0.002139874739039649
